In [1]:
from scipy.io import loadmat
import pandas as pd
import os
import matplotlib.pyplot as plt
import numpy as np
from mapd import Trial, Table, Sinq
import h5py

from matplotlib.figure import Figure
from matplotlib.backends.backend_agg import FigureCanvasAgg as FigureCanvas
%load_ext autoreload
%autoreload 2
%matplotlib widget

def refresh_table_addons():
    import importlib
    import mapd.table_plotters as tps
    import mapd.table_scalars as tbs
    importlib.reload(tps)
    importlib.reload(tbs)

    for name in dir(tps):
        if name.startswith('_'):
            continue
        attr = getattr(tps, name)
        setattr(Table, name[len("plot_"):], attr)
    
    for name in dir(tbs):
        if name.startswith('_'):
            continue
        if name.startswith('compute_'):
            attr = getattr(tbs, name)
            setattr(Table, name[len("compute_"):], attr)

# Create sinq

In [2]:
sinq = Sinq(sinqname='Figure1')
print(sinq.__repr__())
sinq.df

Found data directory: D:\Data
Sinq(Figure1, 20 rows x 29 params): 0 empties; ['210302_F1_C2', '210331_F2_C1', '210405_F1_C1', '210604_F1_C1', '210903_F3_C1', '210915_F1_C1', '210917_F2_C1', '250226_F2_C1', '250227_F2_C1', '250227_F3_C1', '250228_F1_C1', '250228_F2_C1', '250304_F2_C1', '250304_F3_C1', '250425_F1_C1', '250506_F2_C1', '250514_F1_C1', '250923_F2_C1', '250924_F1_C1', '250924_F2_C1'] x ['parquet', 'Table', 'genotype', 'duration', 'rms_velocity', 'outcome_fractions_no_as_no_mv', 'outcome_fractions_as_off', 'outcome_fractions_rest', 'outcome_fractions_no_as_mv', 'outcome_fractions_probe', 'outcome_fractions_as_off_late', 'outcome_fractions_timeout_fail', 'outcome_fractions_timeout', 'outcome_fractions_info', 'lo_state_median_position', 'hi_state_median_position', 'hi_lo_shift', 'hi_state_on_target', 'lo_state_on_target', 'num_trials', 'blue_fraction', 'blue_toggle_fraction', 'most_common_fiberLED', 'lo_target_off_state', 'hi_target_off_state', 'probe_positive_effort', 'success

,parquet,Table,genotype,duration,rms_velocity,outcome_fractions_no_as_no_mv,outcome_fractions_as_off,outcome_fractions_rest,outcome_fractions_no_as_mv,outcome_fractions_probe,...,num_trials,blue_fraction,blue_toggle_fraction,most_common_fiberLED,lo_target_off_state,hi_target_off_state,probe_positive_effort,successes,hard_successes,holding_cost
210302_F1_C2,LEDFlashWithPiezoCueControl_210302_F1_C2_Table...,None,Hot-Cell-Gal4 (test),6836.72718,32982.968041,0.561250,0.173750,0.118750,0.111250,0.000000,...,800.0,1.000000,0.000000,625_red,0.055824,0.082934,8.684294e+05,78.0,37.0,9.147350e+06
210331_F2_C1,LEDFlashWithPiezoCueControl_210331_F2_C1_Table...,None,Hot-Cell-LexA_Chr;81A06_pJFRC7,8428.02290,142491.573361,0.337240,0.188802,0.117188,0.088542,0.000000,...,768.0,1.000000,0.218289,625_red,0.351254,0.070320,3.672936e+06,39.0,13.0,9.204896e+06
210405_F1_C1,LEDFlashWithPiezoCueControl_210405_F1_C1_Table...,None,Hot-Cell-LexA_Chr;81A06_pJFRC7,6582.81488,34006.653998,0.462382,0.087774,0.112853,0.105016,0.000000,...,638.0,1.000000,0.171378,625_red,0.145846,0.206347,1.349486e+06,34.0,13.0,1.307052e+07
210604_F1_C1,LEDFlashWithPiezoCueControl_210604_F1_C1_Table...,None,Hot-Cell-LexA_Chr;35c09_pJFRC7,3685.79266,44418.935879,0.565714,0.102857,0.120000,0.085714,0.000000,...,350.0,1.000000,0.107143,625_red,0.138731,0.180145,1.728146e+06,15.0,9.0,6.519351e+06
210903_F3_C1,LEDFlashTriggerPiezoControl_210903_F3_C1_Table...,None,Hot-Cell-LexA_Chr;35C09_pJFRC7,6303.51030,45596.022785,0.426667,0.131667,0.120000,0.086667,0.000000,...,600.0,1.000000,0.204545,625_red,0.125386,0.072073,1.898442e+06,37.0,14.0,1.363618e+07
210915_F1_C1,LEDFlashTriggerPiezoControl_210915_F1_C1_Table...,None,Hot-Cell-LexA_Chr;78E05_pJFRC7,5857.48988,45659.433547,0.410463,0.084507,0.120724,0.078471,0.000000,...,497.0,1.000000,0.237986,625_red,0.109020,0.279235,2.327835e+06,27.0,12.0,1.325873e+07
210917_F2_C1,LEDFlashTriggerPiezoControl_210917_F2_C1_Table...,None,Hot-Cell-LexA_Chr;31H05_pJFRC7,5780.00094,74683.629443,0.315498,0.276753,0.121771,0.127306,0.000000,...,542.0,1.000000,0.182773,625_red,0.111971,0.241052,4.262578e+06,50.0,19.0,1.208222e+07
250226_F2_C1,LEDFlashTriggerPiezoControl_250226_F2_C1_Table...,None,SS61350_pJFRC7,9447.21096,506486.281215,0.261153,0.455930,0.110990,0.125136,0.014146,...,919.0,0.751904,1.000000,epi_only,0.238928,0.056086,2.533657e+07,93.0,29.0,2.928704e+07
250227_F2_C1,LEDFlashTriggerPiezoControl_250227_F2_C1_Table...,None,SS61350_pJFRC7,7224.62008,236730.246194,0.478431,0.315033,0.117647,0.054902,0.014379,...,765.0,0.666667,1.000000,epi_only,0.115984,0.042458,7.757883e+06,44.0,29.0,1.284016e+07
250227_F3_C1,LEDFlashTriggerPiezoControl_250227_F3_C1_Table...,None,SS61350_pJFRC7,6078.90756,392467.613033,0.360502,0.333856,0.206897,0.056426,0.012539,...,638.0,1.000000,1.000000,epi_only,0.048098,0.169664,1.715732e+07,27.0,12.0,1.004275e+07


# Whole cell recordings

### 210302_F1_C2

In [ ]:
# tr = Trial(fn='LEDFlashWithPiezoCueControl_Raw_210302_F1_C2_1.mat')
# tr.params
# tr = Trial(fn='LEDFlashWithPiezoCueControl_Raw_210302_F1_C2_1.mat')
T = Table('LEDFlashWithPiezoCueControl_210302_F1_C2_Table.parquet',progress_bar=True)
# T.assign_column_value('filtercube_status','blue',trial_min =1, write_to_hdf5=True)
# T.assign_column_value('fiberLED','625_red',trial_min =1,write_to_hdf5=True)
T.extract_trial_properties()
# fig,ax = T.plot_outcomes(savefig=True,format='png')
# fig,ax,df = T.plot_probe_position_heatmap(cmin=-500,cmax=10,format='png')
sinq.add_table(table=T)
sinq.df

### 210319_F2_C1 - Do not use
What is the deal!? There is an enormous number of points and the distributions is near probe_Zero, but the heatmap shows displacement. Not clear where this is coming from...?

In [ ]:
T = Table('LEDFlashWithPiezoCueControl_210319_F2_C1_Table.parquet',progress_bar=True,add_probestate=False)
# # Found 4 target positions - Counter({('lo', 730.0, -180.0, 60.0): 251, ('hi', 730.0, -260.0, 60.0): 199, ('no_state', 0.0, 520.0, 80.0): 50, ('no_state', 0.0, 450.0, 80.0): 50})
# # T.df.loc[T.df['pyasXPosition']==490,'Trial'].index
# # T.exclude_list_of_trials(T.df.loc[T.df['pyasXPosition']==490,'Trial'].index,reason='unused target position')
# # T.df.loc[T.df['pyasXPosition']==520,'probeZero'] = 730
# # T.df.loc[T.df['pyasXPosition']==450,'probeZero'] = 730
# # T.df.loc[T.df['pyasXPosition']==520,'pyasState'] = 'lo'
# # T.df.loc[T.df['pyasXPosition']==450,'pyasState'] = 'hi'
# # T.df.loc[T.df['pyasXPosition']==450,'pyasState']
# # T.write_column_to_trial_files('pyasState')
# # T.write_column_to_trial_files('probeZero')
# T.assign_column_value('filtercube_status','blue',trial_min =1, write_to_hdf5=True)
# T.assign_column_value('fiberLED','625_red',trial_min =1,write_to_hdf5=True)
T.extract_trial_properties()
fig,ax = T.plot_outcomes(savefig=True,format='png')
fig,ax,df = T.plot_probe_position_heatmap(cmin=-500,cmax=10,format='png')
# trial = Trial('LEDFlashWithPiezoCueControl_Raw_210319_F2_C1_1.mat')
# trial.fiberLED

### 210331_F2_C1 - HC-LexA/13XLexAop-ChrimsonR;81A06/pJFRC7 100uL ATR mixed in

In [ ]:
T = Table('LEDFlashWithPiezoCueControl_210331_F2_C1_Table.parquet',progress_bar=True,add_probestate=True)
# T.assign_column_value('filtercube_status','blue',trial_min =1, write_to_hdf5=True)
# T.assign_column_value('fiberLED','625_red',trial_min =1,write_to_hdf5=True)
T.extract_trial_properties()
T.fig_folder = './figpanels'
fig,ax = T.plot_outcomes(savefig=True,format='png')
fig,ax,df = T.plot_probe_position_heatmap(cmin=-500,cmax=10,format='png')
# T.df.loc[775,'Trial'].exclude(reason='end_of_data_poorly_conditioned')


### 210405_F1_C1

In [ ]:
T = Table('LEDFlashWithPiezoCueControl_210405_F1_C1_Table.parquet',progress_bar=True,add_probestate=True)
# T.assign_column_value('filtercube_status','blue',trial_min =1, write_to_hdf5=True)
# T.assign_column_value('fiberLED','625_red',trial_min =1,write_to_hdf5=True)
T.extract_trial_properties()
T.fig_folder = './figpanels'
fig,ax = T.plot_outcomes(savefig=True,format='png')
fig,ax,df = T.plot_probe_position_heatmap(cmin=-500,cmax=10,format='png')
sinq.add_table(table=T)
sinq.df

### 210602_F1_C1 - Do not use
HC-LexA/13XLexAop-ChrimsonR;35C09/pJFRC7 75 ATR mixed in, no attenuator on LED

In [ ]:
T = Table('LEDFlashWithPiezoCueControl_210602_F1_C1_Table.parquet',progress_bar=True,add_probestate=True)
# Found 7 target positions - Counter({(490.0, 60.0): 241, (390.0, 60.0): 200, (310.0, 60.0): 101, (410.0, 60.0): 51, (380.0, 60.0): 50, (240.0, 60.0): 7, (140.0, 60.0): 1})
# # T.df.loc[T.df['pyasXPosition']==490,'Trial'].index
# # T.exclude_list_of_trials(T.df.loc[T.df['pyasXPosition']==490,'Trial'].index,reason='unused target position')
# # T.df.loc[T.df['pyasXPosition']==520,'probeZero'] = 730
# # T.df.loc[T.df['pyasXPosition']==450,'probeZero'] = 730
# # T.df.loc[T.df['pyasXPosition']==520,'pyasState'] = 'lo'
# # T.df.loc[T.df['pyasXPosition']==450,'pyasState'] = 'hi'
# # T.df.loc[T.df['pyasXPosition']==450,'pyasState']
# # T.write_column_to_trial_files('pyasState')
# # T.write_column_to_trial_files('probeZero')

# T.assign_column_value('filtercube_status','blue',trial_min =1, write_to_hdf5=True)
# T.assign_column_value('fiberLED','625_red',trial_min =1,write_to_hdf5=True)

# trial = Trial('LEDFlashWithPiezoCueControl_Raw_210602_F1_C1_1.mat')
# trial.fiberLED

In [ ]:
T.df.loc[T.df.loc[:,'pyasXPosition']==310].index

In [ ]:
T.fig_folder = './figpanels'
T.extract_trial_properties()
fig,ax,df = T.plot_probe_position_heatmap(index=T.df.index,cmin=-500,cmax=10,format='png')
fig

In [ ]:
T.fig_folder = './figpanels'
print('Table: {}'.format(T))
lo_fig,ax = T.plot_probe_distribution(df_filter={'pyasState':'lo'}, index=T.df.index, bin_min=-500, bin_max=10,format='png',export=False)
hi_fig,ax = T.plot_probe_distribution(df_filter={'pyasState':'hi'}, index=T.df.index, bin_min=-500, bin_max=10,format='png',export=False)
hi_fig

### 210604_F1_C1 - HC-LexA/13XLexAop-ChrimsonR;35C09/pJFRC7 75 ATR mixed in

In [ ]:
T = Table('LEDFlashWithPiezoCueControl_210604_F1_C1_Table.parquet',progress_bar=True,add_probestate=True)
# Found 7 target positions - Counter({(490.0, 60.0): 241, (390.0, 60.0): 200, (310.0, 60.0): 101, (410.0, 60.0): 51, (380.0, 60.0): 50, (240.0, 60.0): 7, (140.0, 60.0): 1})
# # T.df.loc[T.df['pyasXPosition']==490,'Trial'].index
# # T.exclude_list_of_trials(T.df.loc[T.df['pyasXPosition']==490,'Trial'].index,reason='unused target position')
# # T.df.loc[T.df['pyasXPosition']==520,'probeZero'] = 730
# # T.df.loc[T.df['pyasXPosition']==450,'probeZero'] = 730
# # T.df.loc[T.df['pyasXPosition']==520,'pyasState'] = 'lo'
# # T.df.loc[T.df['pyasXPosition']==450,'pyasState'] = 'hi'
# # T.df.loc[T.df['pyasXPosition']==450,'pyasState']
# # T.write_column_to_trial_files('pyasState')
# # T.write_column_to_trial_files('probeZero')

T.assign_column_value('filtercube_status','blue',trial_min =1, write_to_hdf5=True)
T.assign_column_value('fiberLED','625_red',trial_min =1,write_to_hdf5=True)

trial = Trial('LEDFlashWithPiezoCueControl_Raw_210604_F1_C1_1.mat')
trial.fiberLED

In [ ]:
T.fig_folder = './figpanels'
T.extract_trial_properties()
fig,ax,df = T.plot_probe_position_heatmap(index=T.df.index,cmin=-500,cmax=10,format='png')
fig

In [ ]:
T.fig_folder = './figpanels'
print('Table: {}'.format(T))
lo_fig,ax = T.plot_probe_distribution(df_filter={'pyasState':'lo'}, index=T.df.index, bin_min=-500, bin_max=10,format='png',export=False)
hi_fig,ax = T.plot_probe_distribution(df_filter={'pyasState':'hi'}, index=T.df.index, bin_min=-500, bin_max=10,format='png',export=False)
hi_fig

### 210903_F3_C1 - HC-LexA/13XLexAop-ChrimsonR;35C09/pJFRC7 75 ATR mixed in

In [ ]:
T = Table('LEDFlashTriggerPiezoControl_210903_F3_C1_Table.parquet',progress_bar=True,add_probestate=True)

T.assign_column_value('filtercube_status','blue',trial_min =1, write_to_hdf5=True)
T.assign_column_value('fiberLED','625_red',trial_min =1,write_to_hdf5=True)

trial = Trial('LEDFlashWithPiezoCueControl_Raw_210602_F1_C1_1.mat')
trial.fiberLED

In [ ]:
T.fig_folder = './figpanels'
T.extract_trial_properties()
fig,ax,df = T.plot_probe_position_heatmap(index=T.df.index,cmin=-500,cmax=10,format='png')
fig

In [ ]:
T.fig_folder = './figpanels'
print('Table: {}'.format(T))
lo_fig,ax = T.plot_probe_distribution(df_filter={'pyasState':'lo'}, index=T.df.index, bin_min=-500, bin_max=10,format='png',export=False)
hi_fig,ax = T.plot_probe_distribution(df_filter={'pyasState':'hi'}, index=T.df.index, bin_min=-500, bin_max=10,format='png',export=False)
hi_fig

### 210915_F1_C1 - HC-LexA/13XLexAop-ChrimsonR;35C09/pJFRC7 75 ATR mixed in

In [ ]:
# T = Table('LEDFlashTriggerPiezoControl_210915_F1_C1_Table.parquet',progress_bar=True,add_probestate=True)
# Found 5 target positions - Counter({(432.0, 60.0): 208, (332.0, 60.0): 189, (2.0, 0.0): 50, (238.0, 60.0): 50, (338.0, 60.0): 50})

# T.exclude_list_of_trials(index = T.df.loc[T.df['pyasXPosition']==2,'Trial'].index,reason='unused target position')
# T.df.loc[T.df['pyasXPosition']==238,'Trial'].index
# T.df.loc[T.df['pyasXPosition']==238,'probeZero'] = 518
# T.df.loc[T.df['pyasXPosition']==338,'probeZero'] = 518
# T.df.loc[T.df['pyasXPosition']==338,'pyasState'] = 'lo'
# T.df.loc[T.df['pyasXPosition']==238,'pyasState'] = 'hi'
# T.df.loc[T.df['pyasXPosition']==338,'pyasState']

# T.df.loc[T.df['pyasXPosition']==432.0,'probeZero'] = 612
# T.df.loc[T.df['pyasXPosition']==332.0,'probeZero'] = 612
# T.df.loc[T.df['pyasXPosition']==432.0,'pyasState'] = 'lo'
# T.df.loc[T.df['pyasXPosition']==332.0,'pyasState'] = 'hi'
# T.df.loc[T.df['pyasXPosition']==338,'pyasState']
# T.write_column_to_trial_files('pyasState')
# T.write_column_to_trial_files('probeZero')

# T.assign_column_value('filtercube_status','blue',trial_min =1, write_to_hdf5=True)
# T.assign_column_value('fiberLED','625_red',trial_min =1,write_to_hdf5=True)

# trial = Trial('LEDFlashWithPiezoCueControl_Raw_210602_F1_C1_1.mat')
# trial.fiberLED

T = sinq.df.loc['210915_F1_C1','Table']
T

In [ ]:
index = T.df.loc[T.df['probeZero']==612.0].index

In [ ]:
T.fig_folder = './figpanels'
T.extract_trial_properties()
fig,ax,df = T.plot_probe_position_heatmap(index=T.df.loc[T.df['probeZero']==612.0].index,cmin=-500,cmax=10,format='png')
fig

In [ ]:
T.fig_folder = './figpanels'
print('Table: {}'.format(T))
# lo_fig,ax = T.plot_probe_distribution(df_filter={'pyasState':'lo'}, index=T.df.index, bin_min=-500, bin_max=10,format='png',export=False)
# hi_fig,ax = T.plot_probe_distribution(df_filter={'pyasState':'hi'}, index=T.df.index, bin_min=-500, bin_max=10,format='png',export=False)
lo_fig,ax = T.plot_probe_distribution(df_filter={'pyasState':'lo'}, index=index, bin_min=-500, bin_max=10,format='png',export=False)
hi_fig,ax = T.plot_probe_distribution(df_filter={'pyasState':'hi'}, index=index, bin_min=-500, bin_max=10,format='png',export=False)
hi_fig

In [ ]:
lo_fig

### 210917_F2_C1 - HC-LexA/13XLexAop-ChrimsonR;35C09/pJFRC7 75 ATR mixed in

In [ ]:
T = Table('LEDFlashTriggerPiezoControl_210917_F2_C1_Table.parquet',progress_bar=True,add_probestate=True)

T.assign_column_value('filtercube_status','blue',trial_min =1, write_to_hdf5=True)
T.assign_column_value('fiberLED','625_red',trial_min =1,write_to_hdf5=True)

trial = Trial('LEDFlashWithPiezoCueControl_Raw_210602_F1_C1_1.mat')
trial.fiberLED

In [ ]:
T.fig_folder = './figpanels'
T.extract_trial_properties()
fig,ax,df = T.plot_probe_position_heatmap(index=T.df.index,cmin=-500,cmax=10,format='png')
fig

In [ ]:
T.fig_folder = './figpanels'
print('Table: {}'.format(T))
lo_fig,ax = T.plot_probe_distribution(df_filter={'pyasState':'lo'}, index=T.df.index, bin_min=-500, bin_max=10,format='png',export=False)
hi_fig,ax = T.plot_probe_distribution(df_filter={'pyasState':'hi'}, index=T.df.index, bin_min=-500, bin_max=10,format='png',export=False)
hi_fig

In [ ]:
lo_fig

# W+ Blue to Green

SS61350>w+;;pJFRC7 = +;pJFRC7/VT057379-p65ADZp in attP40; R20C08-ZpGdbd in attP2 (SS61350)

### 250226_F2_C1 - SS61350>w+;;pJFRC7

Nice fly_cell. There was the strange anticipation of the target change. There was a very long rest before switching to green LED. But even then 

In [ ]:
T = Table('LEDFlashTriggerPiezoControl_250226_F2_C1_Table.parquet')
# T.open_notes_files()
# T.assign_column_value('filtercube_status','blue',trial_min=1,trial_max=691,write_to_hdf5=True)
# print(T.df['filtercube_status'].value_counts())
# T.assign_column_value('filtercube_status','green',trial_min=692,write_to_hdf5=True)
# print(T.df['filtercube_status'].value_counts())
T.assign_column_value('fiberLED','epi_only',trial_min =1,write_to_hdf5=True)

### 250227_F1_C1 - SS61350>w+;;pJFRC7

Blue then Green. Really nice! Timeouts only begin with the green LED

In [ ]:
T = Table('LEDFlashTriggerPiezoControl_250227_F1_C1_Table.parquet')
# T.open_notes_files()
# T.assign_column_value('filtercube_status','blue',trial_min=1,trial_max=510,write_to_hdf5=True)
# print(T.df['filtercube_status'].value_counts())
# T.assign_column_value('filtercube_status','green',trial_min=511,write_to_hdf5=True)
# print(T.df['filtercube_status'].value_counts())
T.assign_column_value('fiberLED','epi_only',trial_min =1,write_to_hdf5=True)

### 250227_F2_C1 - SS61350>w+;;pJFRC7

Blue then Green. 
Nice learning! Some more timeouts and more as off trials with Green.

In [ ]:
T = Table('LEDFlashTriggerPiezoControl_250227_F2_C1_Table.parquet')
# T.open_notes_files()
# T.assign_column_value('filtercube_status','blue',trial_min=1,trial_max=510,write_to_hdf5=True)
# print(T.df['filtercube_status'].value_counts())
# T.assign_column_value('filtercube_status','green',trial_min=511,write_to_hdf5=True)
# print(T.df['filtercube_status'].value_counts())
T.assign_column_value('fiberLED','epi_only',trial_min =1,write_to_hdf5=True)
print(T.df['fiberLED'].value_counts())

### 250227_F3_C1 - SS61350>w+;;pJFRC7

?

In [ ]:
T = Table('LEDFlashTriggerPiezoControl_250227_F3_C1_Table.parquet')

# T.add_df_category('blue',trial_min=1,trial_max=638,categories='_filtercube_status_cat')
# T.assign_column_value('filtercube_status','blue',trial_min=1,trial_max=638,write_to_hdf5=True)
T.assign_column_value('fiberLED','epi_only',trial_min =1,write_to_hdf5=True)
print(T.df['fiberLED'].value_counts())
# T.open_notes_files()

In [ ]:
T.open_notes_files()

### 250228_F1_C1 - SS61350>w+;;pJFRC7

Blue then Green. 
This one takes a long time to learn in each block. Some more timeouts and more as off trials with Green.

In [ ]:
T = Table('LEDFlashTriggerPiezoControl_250228_F1_C1_Table.parquet')
T.progress_bar = True
# T.assign_column_value('filtercube_status','blue',trial_min=1,trial_max=507,write_to_hdf5=True)
# print(T.df['filtercube_status'].value_counts())
# T.assign_column_value('filtercube_status','green',trial_min=508,write_to_hdf5=True)
print(T.df['filtercube_status'].value_counts())
T.assign_column_value('fiberLED','epi_only',trial_min =1,write_to_hdf5=True)
print(T.df['fiberLED'].value_counts())

### 250304_F3_C1 - SS61350>w+;;pJFRC7

Blue then Green. 


In [ ]:
T = Table('LEDFlashTriggerPiezoControl_250304_F3_C1_Table.parquet')
# T.open_notes_files()
# T.extract_trial_properties()
# val = T.assign_column_value('filtercube_status','blue',trial_min=1,trial_max=1450,write_to_hdf5=True)
T.assign_column_value('fiberLED','epi_only',trial_min =1,write_to_hdf5=True)
print(T.df['fiberLED'].value_counts())

In [ ]:
fig,ax = T.plot_outcomes(savefig=True)
# fname = T.export_some_probe_groups(index=idx, from_zero=True)
# fig = plot_probe_trace_from_pickle(fname)
fig.canvas.draw()
figname =  f"./Figure1/example_outcomes_{T.day}_F{T.fly}_C{T.cell}_{T.genotype}_.svg"
fig.savefig(fname=figname, format='svg')
fig

### 250425_F1_C1 - +;pJFRC7;31H05-Gal4

In [ ]:
T = Table('LEDFlashTriggerPiezoControl_250425_F1_C1_Table.parquet')
# T.assign_column_value('filtercube_status','blue',trial_min=1,write_to_hdf5=True)
# T.open_notes_files()
T.assign_column_value('fiberLED','epi_only',trial_min =1,write_to_hdf5=True)
print(T.df['fiberLED'].value_counts())

### 250923_F1_C1 - +;pJFRC7;31H05-Gal4

In [ ]:
id = '250923_F1_C1'
# sinq.df.drop(index=id)

if id in sinq.df.index:
    T = sinq.restore_table(dayflycell=id,overwrite=True)
    T.extract_trial_properties()
    T.find_successful_trials()
else:
    T = Table(f'LEDFlashTriggerPiezoControl_{id}_Table.parquet')
    T.exclude_trials()
    T.assign_column_value('filtercube_status','blue',trial_min=1, write_to_hdf5=True)
    T.assign_column_value('fiberLED','epi_only',trial_min =1,write_to_hdf5=True)
    T.extract_trial_properties()
    T.find_successful_trials()
    sinq.add_table(table=T)
    sinq.df

fig,ax = T.plot_outcomes(savefig=True,format='png')
fig,ax,df = T.plot_probe_position_heatmap(cmin=-500,cmax=10,format='png')

### 250923_F2_C1 - SS61350>w+;;pJFRC7

In [ ]:
id = '250923_F2_C1'
# sinq.df.drop(index=id)

if id in sinq.df.index:
    T = sinq.restore_table(dayflycell=id,overwrite=True)
    T.extract_trial_properties()
    T.find_successful_trials()
else:
    T = Table(f'LEDFlashTriggerPiezoControl_{id}_Table.parquet')
    T.exclude_trials()
    # T.assign_column_value('filtercube_status','blue',trial_min=1, write_to_hdf5=True)
    # T.assign_column_value('fiberLED','epi_only',trial_min =1,write_to_hdf5=True)
    T.extract_trial_properties()
    T.find_successful_trials()
    sinq.add_table(table=T)
    sinq.df

fig,ax = T.plot_outcomes(savefig=True,format='png')
fig,ax,df = T.plot_probe_position_heatmap(cmin=-500,cmax=10,format='png')

### 250924_F1_C1 - SS61350>w+;;pJFRC7

In [ ]:
id = '250924_F1_C1'

T = Table(f'LEDFlashTriggerPiezoControl_{id}_Table.parquet')

T.assign_column_value('filtercube_status','blue',trial_min=1, write_to_hdf5=True)
T.assign_column_value('fiberLED','epi_only',trial_min =1,write_to_hdf5=True)
T.extract_trial_properties()
T.find_successful_trials()
fig,ax = T.plot_outcomes(savefig=True,format='png')
fig,ax,df = T.plot_probe_position_heatmap(cmin=-500,cmax=10,format='png')
sinq.add_table(table=T)
sinq.df

### 250924_F2_C1 - SS61350>w+;;pJFRC7

Something funny happened here when I tried to exclude certain trials. I ended up excluding some I want to include. I suspect it has more to do with heatmap plotting

In [ ]:
id = '250924_F2_C1'
if id in sinq.df.index:
    T = sinq.restore_table(dayflycell=id)
else:
    T = Table(f'LEDFlashTriggerPiezoControl_{id}_Table.parquet')
    # T.assign_column_value('filtercube_status','blue',trial_min=1, write_to_hdf5=True)
    # T.assign_column_value('fiberLED','epi_only',trial_min =1,write_to_hdf5=True)
    # endofdaytrials = T.df.loc[1093:]
    # T.exclude_list_of_trials(endofdaytrials.index,reason='tired, frustrated poor behavior')
    sinq.add_table(table=T)
    sinq.df

T.extract_trial_properties()
T.find_successful_trials()
fig,ax = T.plot_outcomes(savefig=True,format='png')
fig,ax,df = T.plot_probe_position_heatmap(cmin=-500,cmax=10,format='png')

### 250925_F1_C1 - SS61350>w+;;pJFRC7

In [25]:
id = '250925_F1_C1'
if id in sinq.df.index:
    T = sinq.restore_table(dayflycell=id,overwrite=True)
else:
    T = Table(f'LEDFlashTriggerPiezoControl_{id}_Table.parquet')
    # T.assign_column_value('filtercube_status','blue',trial_min=1, write_to_hdf5=True)
    # T.assign_column_value('fiberLED','epi_only',trial_min =1,write_to_hdf5=True)
    # endofdaytrials = T.df.loc[1093:]
    # T.exclude_list_of_trials(endofdaytrials.index,reason='tired, frustrated poor behavior')
    sinq.add_table(table=T)
T.exclude_trials()
T.extract_trial_properties()
T.find_successful_trials()
sinq.add_table(table=T,overwrite=True)
fig,ax = T.plot_outcomes(savefig=True,format='png')
fig,ax,df = T.plot_probe_position_heatmap(cmin=-500,cmax=10,format='png')

Found data directory: D:\Data
T = pd.read_parquet("D:\\Data\\250925\\250925_F1_C1\\LEDFlashTriggerPiezoControl_250925_F1_C1_Table.parquet")
Getting trials
Excluding trials:
[Trial(trial=934, 250925_F1_C1, dT=3.21936, ex=True)]
Getting all meta keys
['as_outcome']
Found 2 target positions - Counter({('hi', 630.0, -280.0, 60.0): 475, ('lo', 630.0, -180.0, 60.0): 458})
Writing filtercube_status to each trial
Writing fiberLED to each trial
Excluding trials:
[Trial(trial=934, 250925_F1_C1, dT=3.21936, ex=True)]
no_as_no_mv: 294; no_as_mv: 99; as_off: 378; success: 87; hard_success: 40
duration
rms_velocity
outcome_fractions_no_as_no_mv
outcome_fractions_as_off
outcome_fractions_rest
outcome_fractions_no_as_mv
outcome_fractions_probe
outcome_fractions_as_off_late
outcome_fractions_timeout_fail
outcome_fractions_timeout
outcome_fractions_info
lo_state_median_position
hi_state_median_position
hi_lo_shift
hi_state_on_target
lo_state_on_target
num_trials
blue_fraction
blue_toggle_fraction
most_c

posx and posy should be finite values


8
3 of 3 timeouts (100.0%) during blue light
Probe positions dataframe created with 933 trials


In [26]:
sinq.df

,parquet,Table,genotype,duration,rms_velocity,outcome_fractions_no_as_no_mv,outcome_fractions_as_off,outcome_fractions_rest,outcome_fractions_no_as_mv,outcome_fractions_probe,...,num_trials,blue_fraction,blue_toggle_fraction,most_common_fiberLED,lo_target_off_state,hi_target_off_state,probe_positive_effort,successes,hard_successes,holding_cost
210302_F1_C2,LEDFlashWithPiezoCueControl_210302_F1_C2_Table...,None,Hot-Cell-Gal4 (test),6836.72718,32982.968041,0.561250,0.173750,0.118750,0.111250,0.000000,...,800.0,1.000000,0.000000,625_red,0.055824,0.082934,8.684294e+05,78.0,37.0,9.147350e+06
210331_F2_C1,LEDFlashWithPiezoCueControl_210331_F2_C1_Table...,None,Hot-Cell-LexA_Chr;81A06_pJFRC7,8428.02290,142491.573361,0.337240,0.188802,0.117188,0.088542,0.000000,...,768.0,1.000000,0.218289,625_red,0.351254,0.070320,3.672936e+06,39.0,13.0,9.204896e+06
210405_F1_C1,LEDFlashWithPiezoCueControl_210405_F1_C1_Table...,None,Hot-Cell-LexA_Chr;81A06_pJFRC7,6582.81488,34006.653998,0.462382,0.087774,0.112853,0.105016,0.000000,...,638.0,1.000000,0.171378,625_red,0.145846,0.206347,1.349486e+06,34.0,13.0,1.307052e+07
210604_F1_C1,LEDFlashWithPiezoCueControl_210604_F1_C1_Table...,None,Hot-Cell-LexA_Chr;35c09_pJFRC7,3685.79266,44418.935879,0.565714,0.102857,0.120000,0.085714,0.000000,...,350.0,1.000000,0.107143,625_red,0.138731,0.180145,1.728146e+06,15.0,9.0,6.519351e+06
210903_F3_C1,LEDFlashTriggerPiezoControl_210903_F3_C1_Table...,None,Hot-Cell-LexA_Chr;35C09_pJFRC7,6303.51030,45596.022785,0.426667,0.131667,0.120000,0.086667,0.000000,...,600.0,1.000000,0.204545,625_red,0.125386,0.072073,1.898442e+06,37.0,14.0,1.363618e+07
210915_F1_C1,LEDFlashTriggerPiezoControl_210915_F1_C1_Table...,None,Hot-Cell-LexA_Chr;78E05_pJFRC7,5857.48988,45659.433547,0.410463,0.084507,0.120724,0.078471,0.000000,...,497.0,1.000000,0.237986,625_red,0.109020,0.279235,2.327835e+06,27.0,12.0,1.325873e+07
210917_F2_C1,LEDFlashTriggerPiezoControl_210917_F2_C1_Table...,None,Hot-Cell-LexA_Chr;31H05_pJFRC7,5780.00094,74683.629443,0.315498,0.276753,0.121771,0.127306,0.000000,...,542.0,1.000000,0.182773,625_red,0.111971,0.241052,4.262578e+06,50.0,19.0,1.208222e+07
250226_F2_C1,LEDFlashTriggerPiezoControl_250226_F2_C1_Table...,None,SS61350_pJFRC7,9447.21096,506486.281215,0.261153,0.455930,0.110990,0.125136,0.014146,...,919.0,0.751904,1.000000,epi_only,0.238928,0.056086,2.533657e+07,93.0,29.0,2.928704e+07
250227_F2_C1,LEDFlashTriggerPiezoControl_250227_F2_C1_Table...,None,SS61350_pJFRC7,7224.62008,236730.246194,0.478431,0.315033,0.117647,0.054902,0.014379,...,765.0,0.666667,1.000000,epi_only,0.115984,0.042458,7.757883e+06,44.0,29.0,1.284016e+07
250227_F3_C1,LEDFlashTriggerPiezoControl_250227_F3_C1_Table...,None,SS61350_pJFRC7,6078.90756,392467.613033,0.360502,0.333856,0.206897,0.056426,0.012539,...,638.0,1.000000,1.000000,epi_only,0.048098,0.169664,1.715732e+07,27.0,12.0,1.004275e+07


### 250925_F2_C1 - SS61350>w+;;pJFRC7

In [21]:
id = '250925_F2_C1'
if id in sinq.df.index:
    T = sinq.restore_table(dayflycell=id,overwrite=True)
    # endofdaytrials = T.df.loc[1081:]
    # T.exclude_list_of_trials(endofdaytrials.index,reason='tired, frustrated poor behavior')
else:
    T = Table(f'LEDFlashTriggerPiezoControl_{id}_Table.parquet')
    T.assign_column_value('filtercube_status','blue',trial_min=1, write_to_hdf5=True)
    T.assign_column_value('fiberLED','epi_only',trial_min =1,write_to_hdf5=True)
    sinq.add_table(table=T)
    sinq.df
T.exclude_trials()
T.extract_trial_properties()
T.find_successful_trials()
fig,ax = T.plot_outcomes(savefig=True,format='png')
fig,ax,df = T.plot_probe_position_heatmap(cmin=-500,cmax=10,format='png')

Found data directory: D:\Data
T = pd.read_parquet("D:\\Data\\250925\\250925_F2_C1\\LEDFlashTriggerPiezoControl_250925_F2_C1_Table.parquet")
Getting trials
Excluding trials:
[Trial(trial=1081, 250925_F2_C1, dT=9.28406, ex=True), Trial(trial=1082, 250925_F2_C1, dT=8.74886, ex=True), Trial(trial=1083, 250925_F2_C1, dT=13.9238, ex=True), Trial(trial=1084, 250925_F2_C1, dT=7.97552, ex=True), Trial(trial=1085, 250925_F2_C1, dT=14.52376, ex=True), Trial(trial=1086, 250925_F2_C1, dT=14.02486, ex=True), Trial(trial=1087, 250925_F2_C1, dT=15.05054, ex=True), Trial(trial=1088, 250925_F2_C1, dT=16.84924, ex=True), Trial(trial=1089, 250925_F2_C1, dT=14.69386, ex=True), Trial(trial=1090, 250925_F2_C1, dT=10.0607, ex=True), Trial(trial=1091, 250925_F2_C1, dT=9.24424, ex=True), Trial(trial=1092, 250925_F2_C1, dT=8.69954, ex=True), Trial(trial=1093, 250925_F2_C1, dT=10.49978, ex=True), Trial(trial=1094, 250925_F2_C1, dT=10.45776, ex=True), Trial(trial=1095, 250925_F2_C1, dT=30.81528, ex=True), Trial(tr

posx and posy should be finite values


no_as_no_mv: 118; no_as_mv: 124; as_off: 449; success: 110; hard_success: 24
8
61 of 61 timeouts (100.0%) during blue light
Probe positions dataframe created with 1080 trials


## w+ Green to Blue

### 250228_F2_C1 - SS61350>w+;;pJFRC7

Green then Blue


In [ ]:
T = Table('LEDFlashTriggerPiezoControl_250228_F2_C1_Table.parquet')
# T.add_df_category('green',trial_min=51,trial_max=713,categories='_filtercube_status_cat')
# T = sinq.df.loc['250228_F2_C1','Table']
# T.assign_column_value('filtercube_status','green',trial_min=51,trial_max=713,write_to_hdf5=True)
print(T.df['filtercube_status'].value_counts())
T.assign_column_value('fiberLED','epi_only',trial_min =1,write_to_hdf5=True)
print(T.df['fiberLED'].value_counts())

### 250304_F2_C1 - SS61350>w+;;pJFRC7

In [ ]:
T = Table('LEDFlashTriggerPiezoControl_250304_F2_C1_Table.parquet')
# T = sinq.df.loc['250304_F2_C1','Table']
# T.assign_column_value('filtercube_status','green',trial_min=1,trial_max=408,write_to_hdf5=True)
# print(T.df['filtercube_status'].value_counts())
# T.assign_column_value('filtercube_status','blue',trial_min=409,trial_max=513,write_to_hdf5=True)
# print(T.df['filtercube_status'].value_counts())
# T.assign_column_value('fiberLED','epi_only',trial_min =1,write_to_hdf5=True)
# print(T.df['fiberLED'].value_counts())

### 250506_F2_C1 >- +;TH-Gal4;UAS-Kir2.1

In [ ]:
sinq

In [ ]:
T = Table('LEDFlashTriggerPiezoControl_250506_F2_C1_Table.parquet')
# T.open_notes_files()

# T.assign_column_value('filtercube_status','green',trial_min=1, write_to_hdf5=True)
# T.assign_column_value('fiberLED','epi_only',trial_min =1,write_to_hdf5=True)
T.extract_trial_properties()
fig,ax = T.plot_outcomes(savefig=True,format='png')
fig,ax,df = T.plot_probe_position_heatmap(cmin=-500,cmax=10,format='png')

# solution_ran_out = T.df.loc[494:]
# solution_ran_out.index
# T.exclude_list_of_trials(solution_ran_out.index,'solution_ran_out')
sinq.add_table(table=T)
sinq.df

In [ ]:
fig = Figure(figsize = (8,6),dpi=200)
ax = fig.add_subplot(1,1,1)
T.plot_some_probe_groups(T.df.loc[410:430,:].index,ax=ax,force_pos=True)
fig

### 250514_F1_C1 >- +;TH-Gal4;UAS-Kir2.1

In [ ]:
# T = Table('LEDFlashTriggerPiezoControl_250514_F1_C1_Table.parquet')
# # T.open_notes_files()

# # T.assign_column_value('filtercube_status','green',trial_min=1, write_to_hdf5=True)
# # T.assign_column_value('fiberLED','epi_only',trial_min =1,write_to_hdf5=True)
# T.extract_trial_properties()
fig,ax = T.plot_outcomes(savefig=True,format='png')
fig,ax,df = T.plot_probe_position_heatmap(cmin=-500,cmax=10,format='png')

# solution_ran_out = T.df.loc[494:]
# solution_ran_out.index
# T.exclude_list_of_trials(solution_ran_out.index,'solution_ran_out')
sinq.add_table(table=T)
sinq.df

In [ ]:
fig = Figure(figsize = (8,6),dpi=200)
ax = fig.add_subplot(1,1,1)
T.plot_some_probe_groups(T.df.loc[410:430,:].index,ax=ax,force_pos=True)
fig

# Sinqing and plotting

In [ ]:
for id in sinq.df.index:
    T = sinq.restore_table(id)
    T.fig_folder = './figpanels'
    T.extract_trial_properties()
    fig,ax = T.plot_outcomes(savefig=True,format='png')
    # fig,ax,df = T.plot_probe_position_heatmap(cmin=-500,cmax=10,format='png')

In [ ]:
sinq.sync